# Week 6 Lab 05: Evaluating Summarization with ROUGE

**Scenario:** Cordwell Home and Hardware
**Estimated duration:** 120 minutes
**Mode:** fully offline. No API keys, no network access, no Docker services.

You have spent Week 6 so far on classification metrics, where every prediction is simply right or wrong. Summarization breaks that model: a candidate summary can be partially right, phrased differently, too long, or subtly made up. This lab is about measuring generated text with **ROUGE**, understanding what the numbers mean, and, just as importantly, catching the cases where a good number hides a bad summary.

You will:

1. Work with a synthetic corpus of 400 Cordwell internal documents built so that ROUGE behavior is predictable and inspectable.
2. Implement two extractive summarizers: a Lead-2 baseline and a TF-IDF sentence selector.
3. Compute ROUGE-1, ROUGE-2, and ROUGE-L with the `rouge-score` library and interpret the results at corpus level and per example.
4. Discover a famous real world effect: the lead baseline is embarrassingly strong, and corpus averages hide where it falls apart.
5. Construct hallucinated and redundant summaries and watch which ROUGE component catches each one.

### How this lab is graded as you go

A provided check harness scores your work. Running the final cell prints a scoreboard. On a fresh notebook every task reports TODO. That is expected: the notebook opens red and turns green as you complete tasks. Checks never crash your notebook; they degrade to TODO or FAIL messages.

## Learning objectives

By the end of this lab you can:

1. Compute ROUGE-1, ROUGE-2, and ROUGE-L precision, recall, and F-measure for candidate summaries against references using `rouge-score`, with the arguments in the correct order.
2. Explain why ROUGE recall is the traditional headline number for summarization, and why precision is the number that catches hallucination and redundancy.
3. Compare two summarization systems at corpus level, then break the comparison down by a document property to expose failure modes that the averages hide.
4. Demonstrate concretely that a summary can gain ROUGE recall by adding made up content, and that repeating correct content collapses precision.
5. Explain why even a semantically perfect summary in different words does not reach a ROUGE score of 1.0, and what that means for interpreting absolute scores.

## 0. Setup

Everything below runs offline on your Mac. The pinned versions match the cohort stack. The core lab needs only five libraries; PyTorch and sentence-transformers appear only in optional Stretch 2 and are not required.

If you are missing a package, install from the provided requirements file:

```bash
pip install -r requirements.txt
```

Run the next two cells. The first loads libraries and prints versions so you can confirm your environment matches. The second defines the check harness that scores your work.

In [ ]:
%pip install -r requirements.txt

In [1]:
# 0.1 Imports, configuration, and reproducibility
%matplotlib inline

import re
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from rouge_score import rouge_scorer

import sklearn
import rouge_score as rouge_score_pkg

CORPUS_SEED = 1207
NUM_DOCS = 400

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

print("pandas       ", pd.__version__)
print("numpy        ", np.__version__)
print("scikit-learn ", sklearn.__version__)
print("rouge-score  ", getattr(rouge_score_pkg, "__version__", "0.1.2"))

pandas        3.0.2
numpy         2.4.4
scikit-learn  1.8.0
rouge-score   0.1.2


In [2]:
# 0.2 Lab check harness (provided). Run it and move on.
# Zero argument check functions read your work from the notebook namespace.
# Statuses: PASS, FAIL (implemented but wrong), TODO (not implemented yet).
# Checks never crash the notebook; they degrade to TODO or FAIL messages.

CHECK_STATUS = {}

_T1_INPUT = "Aisles close at nine. Restock the paint desk! Did the truck arrive? Final walk."
_T1_EXPECTED = ["Aisles close at nine.", "Restock the paint desk!",
                "Did the truck arrive?", "Final walk."]
_T3_DOC0_EXPECTED = ("A calcium chloride moisture test is required before any "
                     "flooring goes over a concrete slab. Hardwood planks must "
                     "acclimate inside the customer's home for at least forty "
                     "eight hours before installation begins. Luxury vinyl plank "
                     "remains the top recommendation for budget conscious "
                     "customers with pets.")
_T4_EXPECTED = {"r1_precision": 1.0, "r1_recall": 0.5, "r1_f": 0.666667,
                "r2_precision": 1.0, "r2_recall": 0.4, "r2_f": 0.571429,
                "rl_precision": 1.0, "rl_recall": 0.5, "rl_f": 0.666667}
_T5_MEANS = {"A_r1_f": 0.529082, "A_r2_f": 0.362647, "A_rl_f": 0.489342,
             "B_r1_f": 0.539086, "B_r2_f": 0.294001, "B_rl_f": 0.406652}
_T6_SUMMARY = {"rouge1": {"A_mean": 0.529082, "A_std": 0.125185,
                          "B_mean": 0.539086, "B_std": 0.049047},
               "rouge2": {"A_mean": 0.362647, "A_std": 0.106394,
                          "B_mean": 0.294001, "B_std": 0.074525},
               "rougeL": {"A_mean": 0.489342, "A_std": 0.111711,
                          "B_mean": 0.406652, "B_std": 0.068654}}
_T6_LAYOUT = {"buried": {"A_rl_f": 0.394586, "B_rl_f": 0.403015},
              "front_loaded": {"A_rl_f": 0.594072, "B_rl_f": 0.410671}}
_T8_MEANS = {"base": {"r1_precision": 0.508695, "r1_recall": 0.562321,
                      "rl_f": 0.395202},
             "hallucinated": {"r1_precision": 0.310327, "r1_recall": 0.644697,
                              "rl_f": 0.275239},
             "redundant": {"r1_precision": 0.181187, "r1_recall": 0.601163,
                           "rl_f": 0.251781}}


def _report(name, status, msg=""):
    CHECK_STATUS[name] = status
    line = f"[{status}] {name}"
    if msg:
        line += f": {msg}"
    print(line)


def _close(a, b, tol=1e-3):
    try:
        return abs(float(a) - float(b)) <= tol
    except (TypeError, ValueError):
        return False


def _get_global(varname):
    return globals().get(varname, None)


def check_task_1():
    name = "Task 1 split_sentences"
    fn = _get_global("split_sentences")
    if fn is None:
        return _report(name, "TODO", "split_sentences is not defined yet")
    try:
        got = fn(_T1_INPUT)
        got2 = fn("First. Second.  Third.")
    except NotImplementedError:
        return _report(name, "TODO")
    except Exception as e:
        return _report(name, "FAIL", f"raised {type(e).__name__}: {e}")
    if got != _T1_EXPECTED:
        return _report(name, "FAIL", f"expected {_T1_EXPECTED}, got {got}")
    if got2 != ["First.", "Second.", "Third."]:
        return _report(name, "FAIL",
                       "double spaces between sentences should not create "
                       f"empty entries, got {got2}")
    _report(name, "PASS")


def check_task_2():
    name = "Task 2 summarize_lead_k"
    fn = _get_global("summarize_lead_k")
    if fn is None:
        return _report(name, "TODO", "summarize_lead_k is not defined yet")
    try:
        got = fn(_T1_INPUT, max_sentences=2)
        got_short = fn("Only one sentence here.", max_sentences=2)
    except NotImplementedError:
        return _report(name, "TODO")
    except Exception as e:
        return _report(name, "FAIL", f"raised {type(e).__name__}: {e}")
    expected = "Aisles close at nine. Restock the paint desk!"
    if got != expected:
        return _report(name, "FAIL", f"expected {expected!r}, got {got!r}")
    if got_short != "Only one sentence here.":
        return _report(name, "FAIL",
                       "a document shorter than max_sentences should be "
                       f"returned whole, got {got_short!r}")
    _report(name, "PASS")


def check_task_3():
    name = "Task 3 summarize_tfidf"
    fn = _get_global("summarize_tfidf")
    if fn is None:
        return _report(name, "TODO", "summarize_tfidf is not defined yet")
    vec = _get_global("vectorizer")
    df = _get_global("corpus_df")
    if vec is None or df is None:
        return _report(name, "TODO", "run the provided setup cells first")
    try:
        got = fn(df.iloc[0]["text"], vec)
        got_short = fn("Two sentences here. Second one.", vec)
    except NotImplementedError:
        return _report(name, "TODO")
    except Exception as e:
        return _report(name, "FAIL", f"raised {type(e).__name__}: {e}")
    if got != _T3_DOC0_EXPECTED:
        return _report(name, "FAIL",
                       "doc 0 selection is wrong. Check the scoring call, the "
                       "tie rule, and that chosen sentences keep document "
                       f"order. Got: {got[:120]}...")
    if got_short != "Two sentences here. Second one.":
        return _report(name, "FAIL",
                       "a document with max_sentences or fewer sentences "
                       f"should be returned whole, got {got_short!r}")
    _report(name, "PASS")


def check_task_4():
    name = "Task 4 compute_rouge_scores"
    fn = _get_global("compute_rouge_scores")
    if fn is None:
        return _report(name, "TODO", "compute_rouge_scores is not defined yet")
    try:
        got = fn("the cat sat on the mat", "the cat sat")
    except NotImplementedError:
        return _report(name, "TODO")
    except Exception as e:
        return _report(name, "FAIL", f"raised {type(e).__name__}: {e}")
    if not isinstance(got, dict):
        return _report(name, "FAIL", f"expected a dict, got {type(got).__name__}")
    missing = [k for k in _T4_EXPECTED if k not in got]
    if missing:
        return _report(name, "FAIL", f"missing keys: {missing}")
    if _close(got["r1_precision"], 0.5) and _close(got["r1_recall"], 1.0):
        return _report(name, "FAIL",
                       "precision and recall look swapped. scorer.score takes "
                       "the reference (target) FIRST and the candidate "
                       "(prediction) SECOND.")
    bad = [k for k, v in _T4_EXPECTED.items() if not _close(got[k], v)]
    if bad:
        detail = ", ".join(f"{k}={got[k]:.4f} (expected {_T4_EXPECTED[k]:.4f})"
                           for k in bad)
        return _report(name, "FAIL", detail)
    _report(name, "PASS")


def check_task_5():
    name = "Task 5 corpus_with_scores"
    cws = _get_global("corpus_with_scores")
    if cws is None:
        return _report(name, "TODO", "corpus_with_scores is not defined yet")
    if len(cws) != 400:
        return _report(name, "FAIL", f"expected 400 rows, got {len(cws)}")
    if len(cws.columns) < 28:
        return _report(name, "FAIL",
                       f"expected at least 28 columns, got {len(cws.columns)}")
    missing = [c for c in _T5_MEANS if c not in cws.columns]
    if missing:
        return _report(name, "FAIL", f"missing columns: {missing}")
    bad = [c for c, v in _T5_MEANS.items()
           if not _close(cws[c].mean(), v)]
    if bad:
        detail = ", ".join(f"mean {c}={cws[c].mean():.4f} "
                           f"(expected {_T5_MEANS[c]:.4f})" for c in bad)
        return _report(name, "FAIL", detail)
    _report(name, "PASS")


def check_task_6():
    name = "Task 6 summary and layout tables"
    st = _get_global("summary_table")
    lt = _get_global("layout_table")
    if st is None or lt is None:
        return _report(name, "TODO",
                       "summary_table or layout_table is not defined yet")
    for metric, cols in _T6_SUMMARY.items():
        if metric not in st.index:
            return _report(name, "FAIL",
                           f"summary_table is missing index entry {metric!r}")
        for col, v in cols.items():
            if col not in st.columns:
                return _report(name, "FAIL",
                               f"summary_table is missing column {col!r}")
            if not _close(st.loc[metric, col], v):
                return _report(name, "FAIL",
                               f"summary_table[{col!r}] for {metric} is "
                               f"{st.loc[metric, col]:.4f}, expected {v:.4f}")
    for lay, cols in _T6_LAYOUT.items():
        if lay not in lt.index:
            return _report(name, "FAIL",
                           f"layout_table is missing index entry {lay!r}")
        for col, v in cols.items():
            if col not in lt.columns:
                return _report(name, "FAIL",
                               f"layout_table is missing column {col!r}")
            if not _close(lt.loc[lay, col], v):
                return _report(name, "FAIL",
                               f"layout_table[{col!r}] for {lay} is "
                               f"{lt.loc[lay, col]:.4f}, expected {v:.4f}")
    _report(name, "PASS")


def check_task_7():
    name = "Task 7 find_extreme_examples"
    ids = _get_global("inspect_ids")
    cws = _get_global("corpus_with_scores")
    if ids is None:
        return _report(name, "TODO", "inspect_ids is not defined yet")
    if cws is None:
        return _report(name, "TODO", "finish Task 5 first")
    expected_keys = {"A_best", "A_worst", "B_best", "B_worst"}
    if set(ids.keys()) != expected_keys:
        return _report(name, "FAIL",
                       f"expected keys {sorted(expected_keys)}, "
                       f"got {sorted(ids.keys())}")
    targets = {"A_best": ("A_rl_f", "max"), "A_worst": ("A_rl_f", "min"),
               "B_best": ("B_rl_f", "max"), "B_worst": ("B_rl_f", "min")}
    for key, (col, kind) in targets.items():
        doc_id = ids[key]
        rows = cws.loc[cws["doc_id"] == doc_id]
        if len(rows) != 1:
            return _report(name, "FAIL", f"{key}={doc_id} is not a valid doc_id")
        extreme = cws[col].max() if kind == "max" else cws[col].min()
        if not _close(rows.iloc[0][col], extreme, tol=1e-9):
            return _report(name, "FAIL",
                           f"{key}={doc_id} has {col}={rows.iloc[0][col]:.4f}, "
                           f"but the corpus {kind} is {extreme:.4f}")
    _report(name, "PASS")


def check_task_8():
    name = "Task 8 build_variant_scores"
    hdf = _get_global("halluc_df")
    if hdf is None:
        return _report(name, "TODO", "halluc_df is not defined yet")
    if len(hdf) != 24:
        return _report(name, "FAIL", f"expected 24 rows, got {len(hdf)}")
    needed = {"doc_id", "variant", "r1_precision", "r1_recall",
              "rl_recall", "rl_f"}
    missing = needed - set(hdf.columns)
    if missing:
        return _report(name, "FAIL", f"missing columns: {sorted(missing)}")
    if set(hdf["variant"].unique()) != {"base", "hallucinated", "redundant"}:
        return _report(name, "FAIL",
                       "variant values must be base, hallucinated, redundant, "
                       f"got {sorted(hdf['variant'].unique())}")
    grouped = hdf.groupby("variant")[["r1_precision", "r1_recall", "rl_f"]].mean()
    for variant, cols in _T8_MEANS.items():
        for col, v in cols.items():
            if not _close(grouped.loc[variant, col], v):
                return _report(name, "FAIL",
                               f"mean {col} for {variant} is "
                               f"{grouped.loc[variant, col]:.4f}, "
                               f"expected {v:.4f}")
    _report(name, "PASS")


ALL_CHECKS = [check_task_1, check_task_2, check_task_3, check_task_4,
              check_task_5, check_task_6, check_task_7, check_task_8]


def run_all_checks():
    CHECK_STATUS.clear()
    for check in ALL_CHECKS:
        check()
    passes = sum(1 for s in CHECK_STATUS.values() if s == "PASS")
    print()
    print(f"Scoreboard: {passes} of {len(ALL_CHECKS)} checks passing.")


print("Check harness loaded. Run run_all_checks() any time.")

Check harness loaded. Run run_all_checks() any time.


## 1. The Cordwell corpus: built so ROUGE behavior is inspectable

Real summarization corpora are messy in ways that make a first ROUGE lab confusing. This corpus is synthetic and engineered so that every effect you measure has a visible cause. Three design choices matter:

**Key facts versus filler.** Every document mixes three department specific key facts (the content a good summary should carry) with four to six generic operational filler sentences that repeat across the whole corpus. Because filler repeats everywhere, a corpus level TF-IDF fit assigns filler words low weight and key fact words high weight. That is the signal System B will exploit.

**Two layouts.** Half the documents are `front_loaded` (key facts right after the opening sentence) and half are `buried` (key facts at the end, after the filler). A summarizer that trusts the top of the document will behave very differently on the two layouts. The corpus records which layout each document uses, which lets you group results later.

**Paraphrased references.** Each reference summary contains an opener plus reworded versions of the first two key facts. The reference shares content words with the document but not exact sentences. This mirrors reality: human reference summaries almost never quote the source verbatim, and it is exactly why ROUGE scores rarely approach 1.0.

The generator also records an `oracle_summary` for each document: the same content as the reference, expressed in the document's own words. Ignore it for now. It becomes the star of Stretch 1.

Run the next two cells. Skim the key fact pairs as they scroll past: each pair is the document phrasing and the reference phrasing of the same fact.

In [3]:
# 1.1 Corpus vocabulary: key facts, filler, and templates (provided)

DEPARTMENTS = [
    "Flooring", "Paint", "Kitchen", "Garden",
    "Electrical", "Plumbing", "Installation Services", "Customer Service",
]

DOC_TYPES = [
    "policy update", "associate training memo", "installation checklist",
    "product troubleshooting guide", "promotion briefing", "safety reminder",
]

# Each key fact is a pair: (sentence as written in the document,
# reworded sentence as it appears in the reference summary).
KEY_FACTS = {
    "Flooring": [
        ("Hardwood planks must acclimate inside the customer's home for at least forty eight hours before installation begins.",
         "Hardwood planks need forty eight hours to acclimate in the home before installation."),
        ("A calcium chloride moisture test is required before any flooring goes over a concrete slab.",
         "Concrete slabs require a moisture test before flooring installation."),
        ("Associates must walk customers through the difference between glue down and floating floor systems.",
         "Associates should explain glue down versus floating floor systems to customers."),
        ("Luxury vinyl plank remains the top recommendation for budget conscious customers with pets.",
         "Luxury vinyl plank is the go to recommendation for budget conscious pet owners."),
        ("Underlayment selection depends on the subfloor type and must be confirmed before checkout.",
         "Confirm underlayment choice against the subfloor type before checkout."),
        ("Transition strips and stair nosing should be quoted with every flooring estimate.",
         "Every flooring estimate should include transition strips and stair nosing."),
    ],
    "Paint": [
        ("Tinting mistakes must be logged in the paint desk error book before the end of the shift.",
         "Log every tinting mistake in the paint desk error book by end of shift."),
        ("Bare drywall always needs a dedicated primer coat before the finish color is applied.",
         "Bare drywall requires a primer coat before any finish color."),
        ("Satin and semi gloss sheens are the standard recommendation for kitchens and bathrooms.",
         "Recommend satin or semi gloss sheens for kitchens and bathrooms."),
        ("Low VOC product lines should be offered first to customers sensitive to paint odor.",
         "Offer low VOC lines first to odor sensitive customers."),
        ("Five gallon buckets qualify for contractor pricing when the account is registered.",
         "Registered contractor accounts get contractor pricing on five gallon buckets."),
        ("Color match scans require a chip at least one inch square for accurate results.",
         "Color match scans need a paint chip at least one inch square."),
    ],
    "Kitchen": [
        ("Cabinet lead times can stretch past ten weeks during the spring remodeling season.",
         "Cabinet lead times may exceed ten weeks in the spring remodeling season."),
        ("Every design appointment must be logged in the Cordwell design center calendar.",
         "Design appointments belong in the Cordwell design center calendar."),
        ("Countertop templating is scheduled only after cabinets are fully installed and level.",
         "Countertop templating happens only after cabinets are installed and level."),
        ("Customers financing cabinets and countertops together qualify for the bundled project rate.",
         "Bundled financing of cabinets and countertops qualifies for the project rate."),
        ("Associates must document job site access, parking, and stair constraints on the intake form.",
         "Record job site access, parking, and stair constraints on the intake form."),
        ("Appliance delivery dates should be coordinated with the installation team before ordering.",
         "Coordinate appliance delivery dates with the installation team before ordering."),
    ],
    "Garden": [
        ("Outdoor power equipment may only be fueled and tested in the designated outdoor bay.",
         "Fuel and test outdoor power equipment only in the designated outdoor bay."),
        ("Frost warnings must be shared with any customer buying tender annuals in early spring.",
         "Warn customers buying tender annuals about early spring frost."),
        ("Mower and trimmer warranties must be registered at checkout to stay valid.",
         "Register mower and trimmer warranties at checkout to keep them valid."),
        ("Seasonal endcaps should prioritize soil, mulch, and the week's top selling plants.",
         "Stock seasonal endcaps with soil, mulch, and top selling plants."),
        ("Pesticide sales require the associate to point out the label's application instructions.",
         "Point out label application instructions with every pesticide sale."),
        ("Live goods deliveries are inspected and watered within two hours of arrival.",
         "Inspect and water live goods deliveries within two hours of arrival."),
    ],
    "Electrical": [
        ("Associates may explain product features but must never give specific wiring advice.",
         "Associates can explain product features but never give wiring advice."),
        ("GFCI outlets are the standard recommendation for kitchens, bathrooms, and outdoor circuits.",
         "Recommend GFCI outlets for kitchens, bathrooms, and outdoor circuits."),
        ("Customers should be reminded that permits and licensed electricians are required for panel work.",
         "Panel work requires permits and a licensed electrician."),
        ("Lighting recommendations should cover lumens, color temperature, and fixture rating.",
         "Cover lumens, color temperature, and fixture rating in lighting recommendations."),
        ("Recalled electrical items must be pulled from the shelf and scanned out the same day.",
         "Pull and scan out recalled electrical items the same day."),
        ("Extension cords are rated by gauge and length, and associates must match the cord to the load.",
         "Match extension cord gauge and length to the customer's load."),
    ],
    "Plumbing": [
        ("Water heater replacements may need a municipal permit depending on the customer's city.",
         "Water heater replacements can require a municipal permit."),
        ("Toilet installations should always be quoted with a new wax ring, supply line, and shutoff valve.",
         "Quote toilet installations with a new wax ring, supply line, and shutoff valve."),
        ("Associates must not diagnose plumbing failures and should refer customers to licensed plumbers.",
         "Refer plumbing failure diagnosis to licensed plumbers."),
        ("Dye tablets are the recommended first step for customers chasing a running toilet.",
         "Recommend dye tablets as the first step for a running toilet."),
        ("PEX and copper fittings are stocked in separate aisles and must not be mixed in bins.",
         "Keep PEX and copper fittings in separate bins and aisles."),
        ("Sump pump returns require a water test in the returns bay before restocking.",
         "Water test returned sump pumps before restocking."),
    ],
    "Installation Services": [
        ("Every installation project must be scheduled through the central Cordwell services portal.",
         "Schedule all installation projects through the central Cordwell services portal."),
        ("Customers must receive a written estimate before any installation work is booked.",
         "Provide a written estimate before booking installation work."),
        ("The pre installation checklist must be completed during the in home measurement visit.",
         "Complete the pre installation checklist during the in home measurement."),
        ("Access limitations like narrow staircases or low ceilings must be noted on the work order.",
         "Note access limitations such as narrow staircases on the work order."),
        ("Installer background checks are renewed annually and tracked by the services office.",
         "The services office tracks annual installer background check renewals."),
        ("Post installation walkthroughs require the customer's signature on the completion form.",
         "Collect the customer's signature on the completion form after the walkthrough."),
    ],
    "Customer Service": [
        ("Return windows differ between special order items and standard in stock merchandise.",
         "Special order and in stock items carry different return windows."),
        ("Associates should attempt to resolve concerns directly before escalating to a manager.",
         "Try to resolve customer concerns before escalating to a manager."),
        ("Price adjustments are allowed within fourteen days when a competitor advertises a lower price.",
         "Competitor price adjustments are allowed within fourteen days."),
        ("Damaged merchandise claims must be photographed and tagged before leaving the service desk.",
         "Photograph and tag damaged merchandise claims at the service desk."),
        ("Post checkout surveys feed the store's weekly customer satisfaction scorecard.",
         "Post checkout surveys drive the weekly customer satisfaction scorecard."),
        ("Gift card balance checks do not require a receipt or photo identification.",
         "Gift card balance checks need no receipt or identification."),
    ],
}

# Filler repeats across many documents, so a corpus level TF-IDF fit
# assigns filler words low weight.
FILLER_SENTENCES = [
    "Associates should greet every customer within thirty seconds of entering the department.",
    "Inventory counts must be updated nightly to reflect online and in store sales.",
    "All signage must clearly display the current price and any active promotion.",
    "Weekend traffic peaks between ten in the morning and two in the afternoon.",
    "Same day pickup is available on in stock items ordered before four in the afternoon.",
    "Break schedules are posted in the back office every Sunday evening.",
    "Aisle displays must be faced and restocked before the store opens.",
    "Radios should stay on the department channel during business hours.",
    "Lost and found items are logged at the customer service desk.",
    "Team huddles happen fifteen minutes before every shift change.",
]

SUMMARY_OPENERS = [
    "This {doc_type} covers guidance for the {dept} department at Cordwell Home and Hardware.",
    "This {doc_type} outlines expectations for {dept} associates at Cordwell Home and Hardware.",
    "This {doc_type} summarizes priorities for the {dept} team at Cordwell Home and Hardware.",
]

print(f"{len(DEPARTMENTS)} departments, {sum(len(v) for v in KEY_FACTS.values())} key fact pairs, "
      f"{len(FILLER_SENTENCES)} filler sentences.")

8 departments, 48 key fact pairs, 10 filler sentences.


In [4]:
# 1.2 Corpus generator (provided)

def build_corpus(num_docs: int = NUM_DOCS, seed: int = CORPUS_SEED) -> pd.DataFrame:
    """Generate the Cordwell document corpus with references and metadata.

    Uses a locally seeded random.Random so re-running this cell always
    produces the identical corpus, regardless of what ran before it.
    """
    rng = random.Random(seed)
    rows = []
    for doc_id in range(num_docs):
        dept = rng.choice(DEPARTMENTS)
        doc_type = rng.choice(DOC_TYPES)
        layout = rng.choice(["front_loaded", "buried"])

        facts = rng.sample(KEY_FACTS[dept], k=3)
        key_doc_sentences = [f[0] for f in facts]
        key_summary_sentences = [f[1] for f in facts]

        fillers = rng.sample(FILLER_SENTENCES, k=rng.randint(4, 6))

        opener_template = rng.choice(SUMMARY_OPENERS)
        doc_opening = (
            f"This {doc_type} applies to the {dept} department "
            f"at Cordwell Home and Hardware."
        )

        if layout == "front_loaded":
            body = [doc_opening] + key_doc_sentences + fillers
        else:
            body = [doc_opening] + fillers + key_doc_sentences

        reference = " ".join(
            [opener_template.format(doc_type=doc_type, dept=dept)]
            + key_summary_sentences[:2]
        )
        # Same content as the reference, in the document's own words.
        oracle = " ".join([doc_opening] + key_doc_sentences[:2])

        rows.append({
            "doc_id": doc_id,
            "department": dept,
            "doc_type": doc_type,
            "layout": layout,
            "text": " ".join(body),
            "reference_summary": reference,
            "oracle_summary": oracle,
            "num_sentences": len(body),
        })
    return pd.DataFrame(rows)


corpus_df = build_corpus()
print(f"Generated {len(corpus_df)} documents.")
print(corpus_df["layout"].value_counts().to_dict())
corpus_df[["doc_id", "department", "doc_type", "layout", "num_sentences"]].head()

Generated 400 documents.
{'buried': 210, 'front_loaded': 190}


,doc_id,department,doc_type,layout,num_sentences
0,0,Flooring,policy update,buried,8
1,1,Kitchen,policy update,buried,10
2,2,Plumbing,promotion briefing,front_loaded,8
3,3,Customer Service,installation checklist,front_loaded,9
4,4,Flooring,product troubleshooting guide,front_loaded,9


In [5]:
# 1.3 Look at one document and its reference (provided)

sample = corpus_df.iloc[0]
print(f"doc_id={sample['doc_id']}  dept={sample['department']}  layout={sample['layout']}")
print()
print("DOCUMENT:")
print(sample["text"])
print()
print("REFERENCE SUMMARY:")
print(sample["reference_summary"])

doc_id=0  dept=Flooring  layout=buried

DOCUMENT:
This policy update applies to the Flooring department at Cordwell Home and Hardware. Team huddles happen fifteen minutes before every shift change. Weekend traffic peaks between ten in the morning and two in the afternoon. Inventory counts must be updated nightly to reflect online and in store sales. Aisle displays must be faced and restocked before the store opens. A calcium chloride moisture test is required before any flooring goes over a concrete slab. Hardwood planks must acclimate inside the customer's home for at least forty eight hours before installation begins. Luxury vinyl plank remains the top recommendation for budget conscious customers with pets.

REFERENCE SUMMARY:
This policy update summarizes priorities for the Flooring team at Cordwell Home and Hardware. Concrete slabs require a moisture test before flooring installation. Hardwood planks need forty eight hours to acclimate in the home before installation.


## 2. Two candidate summarization systems

You will build two deliberately simple extractive summarizers. Simple is the point: their failure modes are legible, and ROUGE will react to those failures in ways you can trace.

**System A, Lead-2.** Take the first two sentences of the document. This sounds like a joke baseline. It is not. In news summarization, the lead-3 baseline (first three sentences) famously beat many sophisticated neural systems on ROUGE for years, because journalists front load articles. Watch what it does on our two layouts.

**System B, TF-IDF sentence selector.** Score every sentence by how informative its words are according to a corpus level TF-IDF fit, then keep the top three sentences in their original order. This system does not care where a sentence sits in the document. It only cares whether the words are distinctive.

Both need a sentence splitter first. That is Task 1.

### Task 1: sentence splitter

Implement `split_sentences`. The regex is provided: it breaks after a period, exclamation point, or question mark that is followed by whitespace. Your job is the surrounding logic: split, clean, and filter.

**Worked target output.** Your function must reproduce this exactly:

```text
>>> split_sentences("Aisles close at nine. Restock the paint desk! Did the truck arrive? Final walk.")
['Aisles close at nine.', 'Restock the paint desk!', 'Did the truck arrive?', 'Final walk.']
```

Note the final sentence has no trailing whitespace after its period, and it still appears. Splitting produces it as the last piece.

In [6]:
# Task 1: sentence splitter (STUDENT TODO)

SENTENCE_SPLIT_REGEX = re.compile(r"(?<=[.!?])\s+")


def split_sentences(text):
    """Split text into a clean list of sentences.

    Contract:
      - Split on SENTENCE_SPLIT_REGEX, which breaks after a period,
        exclamation point, or question mark followed by whitespace.
      - Strip surrounding whitespace from each piece.
      - Drop empty strings.
      - Return the sentences as a list of strings in original order.
    """
    raise NotImplementedError("Task 1")


check_task_1()

[TODO] Task 1 split_sentences


### Task 2: System A, the Lead-2 baseline

Implement `summarize_lead_k`. Three lines of thinking: split, slice, join.

**Worked target output.**

```text
>>> summarize_lead_k("Aisles close at nine. Restock the paint desk! Did the truck arrive? Final walk.", max_sentences=2)
'Aisles close at nine. Restock the paint desk!'

>>> summarize_lead_k("Only one sentence here.", max_sentences=2)
'Only one sentence here.'
```

The second case shows the contract when the document is shorter than the requested summary: return everything you have.

In [7]:
# Task 2: System A, Lead-2 baseline (STUDENT TODO)

def summarize_lead_k(text, max_sentences=2):
    """Return the first max_sentences sentences joined by single spaces.

    Contract:
      - Use split_sentences to break the text apart.
      - Keep the first max_sentences sentences in order.
      - If the text has fewer sentences than max_sentences, keep them all.
      - Join the kept sentences with single spaces and return the string.
    """
    raise NotImplementedError("Task 2")


check_task_2()

[TODO] Task 2 summarize_lead_k


### Provided plumbing: the TF-IDF vectorizer and sentence scorer

The next cell is provided, and one line in it deserves your attention because it encodes a real world gotcha.

`TfidfVectorizer` normalizes each output vector to unit length by default. That is the right behavior for cosine similarity, but it silently ruins our use case: if every sentence vector has length 1.0, then an average of its values mostly measures how short the sentence is, not how informative its words are. During development of this lab, the default setting made the summarizer confidently select the most generic filler in every document. Passing `norm=None` keeps raw TF-IDF weights so that informative words genuinely score higher.

The lesson generalizes: library defaults are tuned for the common case, and your case may not be the common case. When a pipeline behaves absurdly, print the intermediate numbers.

`sentence_scores` returns one score per sentence: the mean of the nonzero TF-IDF values for the words in that sentence. Mean, not sum, so long sentences do not win simply by having more words.

In [8]:
# 2.1 TF-IDF vectorizer and sentence scoring helpers (provided)

def make_tfidf_vectorizer(texts):
    """Fit a corpus level TF-IDF vectorizer.

    norm=None is deliberate: the default L2 normalization would scale
    every sentence vector to unit length, turning our mean based score
    into a sentence length artifact instead of a word informativeness
    signal. See the markdown cell above.
    """
    vectorizer = TfidfVectorizer(lowercase=True, stop_words="english", norm=None)
    vectorizer.fit(texts)
    return vectorizer


def sentence_scores(sentences, vectorizer):
    """Score each sentence as the mean of its nonzero TF-IDF values.

    Returns a list of floats, one per sentence, in the same order.
    A sentence with no in-vocabulary words scores 0.0.
    """
    matrix = vectorizer.transform(sentences)
    scores = []
    for i in range(matrix.shape[0]):
        row = matrix.getrow(i)
        if row.nnz == 0:
            scores.append(0.0)
        else:
            scores.append(row.sum() / row.nnz)
    return scores


vectorizer = make_tfidf_vectorizer(corpus_df["text"].tolist())
print(f"Vectorizer fitted on {len(corpus_df)} documents, "
      f"vocabulary size {len(vectorizer.vocabulary_)}.")

Vectorizer fitted on 400 documents, vocabulary size 372.


### Task 3: System B, the TF-IDF sentence selector

Implement `summarize_tfidf`. The vectorizer and scorer are provided; your work is the selection logic: rank sentences by score, keep the top `max_sentences`, and reassemble them **in their original document order** so the summary reads coherently.

The tie rule matters for reproducibility: when two sentences score identically, the earlier one wins. A clean way to get both the ranking and the tie rule in one move is to sort sentence indices by `(-score, index)`.

**Worked target output.** On document 0 (a `buried` layout document), your function must select the three key fact sentences even though they sit at the end of the document:

```text
>>> summarize_tfidf(corpus_df.iloc[0]["text"], vectorizer)
'A calcium chloride moisture test is required before any flooring goes over a concrete slab. Hardwood planks must acclimate inside the customer's home for at least forty eight hours before installation begins. Luxury vinyl plank remains the top recommendation for budget conscious customers with pets.'
```

Compare that to what Lead-2 grabs from the same document, which you saw in section 1.3. Same document, radically different summary.

In [9]:
# Task 3: System B, TF-IDF sentence selector (STUDENT TODO)

def summarize_tfidf(text, vectorizer, max_sentences=3):
    """Return an extractive summary of the highest scoring sentences.

    Contract:
      - Split the text into sentences.
      - If there are max_sentences or fewer, return them all joined by
        single spaces.
      - Otherwise score every sentence with sentence_scores(sentences,
        vectorizer).
      - Select the max_sentences highest scoring sentences. Break score
        ties in favor of the earlier sentence.
      - Reassemble the selected sentences in their original document
        order, joined by single spaces, and return the string.
    """
    raise NotImplementedError("Task 3")


check_task_3()

[TODO] Task 3 summarize_tfidf


In [10]:
# 2.2 Apply both systems across the corpus (provided, guarded)

try:
    corpus_df["candidate_A"] = corpus_df["text"].apply(summarize_lead_k)
    corpus_df["candidate_B"] = corpus_df["text"].apply(
        lambda t: summarize_tfidf(t, vectorizer)
    )
    print("Candidate summaries generated for both systems.")
    print()
    print("Doc 0, System A:", corpus_df.iloc[0]["candidate_A"])
    print()
    print("Doc 0, System B:", corpus_df.iloc[0]["candidate_B"])
except NotImplementedError as exc:
    print(f"Waiting on {exc}: finish Tasks 1 through 3, then re-run this cell.")

Waiting on Task 2: finish Tasks 1 through 3, then re-run this cell.


## 3. Computing ROUGE

Now the metric itself. The `rouge-score` library (Google Research's implementation, the one referenced in your slides) computes three variants:

- **ROUGE-1**: overlap of single words between candidate and reference.
- **ROUGE-2**: overlap of two word sequences. Harder to match by luck, so usually more discriminative.
- **ROUGE-L**: longest common subsequence. Rewards keeping words in the same order without requiring them to be adjacent.

Each variant reports three numbers:

- **Recall**: of the words in the reference, how many did the candidate cover? The traditional headline number for summarization, because a summary's first job is to cover the reference content.
- **Precision**: of the words in the candidate, how many appear in the reference? This is the number to watch for hallucination and padding.
- **F-measure**: harmonic mean of the two. The balanced default for comparing systems.

### The argument order trap

This one causes silent, wrong results in production code, so it gets its own box.

`scorer.score(target, prediction)` takes the **reference first** and the **candidate second**. This is the reverse of what most people guess, and the reverse of sacrebleu's argument order, which you will meet in the BLEU lab. Nothing errors if you swap them. Every precision silently becomes a recall and every recall becomes a precision, and your hallucination analysis quietly inverts. The Task 4 check probes for exactly this mistake with an asymmetric pair, and will tell you if your values look swapped.

### Task 4: the scoring function

Implement `compute_rouge_scores`. The scorer object is created for you in the stub. Call it correctly, then unpack its result into a flat dictionary with nine keys.

The scorer returns a dictionary mapping each metric name to a `Score` object with `.precision`, `.recall`, and `.fmeasure` attributes.

**Worked target output.** For a reference of `the cat sat on the mat` and a candidate of `the cat sat`:

```text
>>> compute_rouge_scores("the cat sat on the mat", "the cat sat")
{'r1_precision': 1.0, 'r1_recall': 0.5, 'r1_f': 0.6667,
 'r2_precision': 1.0, 'r2_recall': 0.4, 'r2_f': 0.5714,
 'rl_precision': 1.0, 'rl_recall': 0.5, 'rl_f': 0.6667}
```

Sanity check the numbers by hand: all three candidate words appear in the reference, so precision is 1.0. The reference has six words and the candidate covers three of them, so recall is 0.5. If your output shows precision 0.5 and recall 1.0, your arguments are swapped.

In [11]:
# Task 4: ROUGE scoring function (STUDENT TODO)

# The scorer is created for you. use_stemmer=True folds word variants
# together (associate and associates count as a match), matching the
# standard reporting configuration.
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"],
                                  use_stemmer=True)


def compute_rouge_scores(reference, candidate):
    """Score one candidate against one reference.

    Contract:
      - Call scorer.score with the REFERENCE as the first argument
        (target) and the CANDIDATE as the second (prediction). This
        order is the trap described above: swapping silently exchanges
        every precision and recall.
      - The call returns a dict mapping "rouge1", "rouge2", "rougeL" to
        Score objects with .precision, .recall, and .fmeasure.
      - Return a flat dict with these nine keys:
          r1_precision, r1_recall, r1_f,
          r2_precision, r2_recall, r2_f,
          rl_precision, rl_recall, rl_f
    """
    raise NotImplementedError("Task 4")


check_task_4()

[TODO] Task 4 compute_rouge_scores


### Task 5: score the whole corpus

Implement `score_corpus`, which runs your scoring function down an entire candidate column and returns the results as a DataFrame with prefixed column names, ready to sit alongside the corpus.

The assembly code below the stub is provided: it scores both systems and concatenates everything into `corpus_with_scores`, the DataFrame the rest of the lab lives on.

**Worked target output.**

```text
corpus_with_scores shape: (400, 28)

First few score columns for doc 0:
A_r1_f = 0.4483   A_rl_f = 0.3793
B_r1_f = 0.5122   B_rl_f = 0.2927
```

The 28 columns are the 10 corpus columns plus 9 score columns per system.

In [12]:
# Task 5: score the whole corpus (STUDENT TODO)

def score_corpus(df, candidate_col, prefix):
    """Score every row's candidate against its reference.

    Contract:
      - For each row of df, call compute_rouge_scores with the row's
        reference_summary and the row's candidate_col value.
      - Rename each returned key by prepending the prefix and an
        underscore (prefix "A" turns r1_f into A_r1_f).
      - Return a DataFrame with one row per input row, carrying the
        same index as df so it concatenates cleanly.
    """
    raise NotImplementedError("Task 5")


# Assembly (provided): scores both systems, builds corpus_with_scores.
try:
    rouge_A = score_corpus(corpus_df, "candidate_A", "A")
    rouge_B = score_corpus(corpus_df, "candidate_B", "B")
    corpus_with_scores = pd.concat([corpus_df, rouge_A, rouge_B], axis=1)
    print("corpus_with_scores shape:", corpus_with_scores.shape)
    row0 = corpus_with_scores.iloc[0]
    print()
    print("First few score columns for doc 0:")
    print(f"A_r1_f = {row0['A_r1_f']:.4f}   A_rl_f = {row0['A_rl_f']:.4f}")
    print(f"B_r1_f = {row0['B_r1_f']:.4f}   B_rl_f = {row0['B_rl_f']:.4f}")
except NotImplementedError as exc:
    print(f"Waiting on {exc}.")
except KeyError:
    print("candidate columns are missing: finish Tasks 1 through 3 and "
          "re-run the apply cell first.")

check_task_5()

Waiting on Task 5.
[TODO] Task 5 corpus_with_scores: corpus_with_scores is not defined yet


## 4. Corpus level comparison

### Task 6: summary tables and the histogram

Time to answer the headline question: which system is better? You will build three views, and they will disagree with each other in an instructive way.

Implement two functions:

**`build_summary_tables`** returns a pair of DataFrames.

The first, `summary_table`, is indexed by `rouge1`, `rouge2`, `rougeL` with columns `A_mean`, `A_std`, `B_mean`, `B_std`, computed from the F-measure columns.

The second, `layout_table`, is the mean of `A_rl_f` and `B_rl_f` grouped by `layout`.

**`plot_rouge_l_histogram`** draws overlapping histograms of `A_rl_f` and `B_rl_f` on one axes, with transparency, axis labels, and a legend, so the two score distributions can be compared by shape.

**Worked target output.** The first row of `summary_table` and the structure of the rest:

```text
         A_mean   A_std  B_mean   B_std
rouge1   0.5291  0.1252  0.5391  0.0490
rouge2      ...     ...     ...     ...
rougeL      ...     ...     ...     ...
```

Before you run the checkpoint discussion, look at your own numbers and form an opinion: which system would you ship?

In [13]:
# Task 6: summary tables and histogram (STUDENT TODO)

F_COLUMNS = {"rouge1": ("A_r1_f", "B_r1_f"),
             "rouge2": ("A_r2_f", "B_r2_f"),
             "rougeL": ("A_rl_f", "B_rl_f")}


def build_summary_tables(scores_df):
    """Return (summary_table, layout_table).

    Contract:
      - summary_table: index rouge1, rouge2, rougeL; columns A_mean,
        A_std, B_mean, B_std; values are the mean and standard
        deviation of the matching F-measure columns in F_COLUMNS.
      - layout_table: the result of grouping scores_df by layout and
        taking the mean of A_rl_f and B_rl_f.
    """
    raise NotImplementedError("Task 6")


def plot_rouge_l_histogram(scores_df):
    """Draw overlapping histograms of A_rl_f and B_rl_f.

    Contract:
      - Both distributions on one axes, using alpha for transparency
        so overlap stays readable.
      - Label the x axis (ROUGE-L F-measure), the y axis (number of
        documents), add a title and a legend naming the systems.
      - Call plt.show() at the end.
    """
    raise NotImplementedError("Task 6")


try:
    summary_table, layout_table = build_summary_tables(corpus_with_scores)
    display(summary_table.round(4))
    display(layout_table.round(4))
    plot_rouge_l_histogram(corpus_with_scores)
except NotImplementedError as exc:
    print(f"Waiting on {exc}.")
except NameError:
    print("Finish Task 5 first: corpus_with_scores is not defined.")

check_task_6()

Finish Task 5 first: corpus_with_scores is not defined.
[TODO] Task 6 summary and layout tables: summary_table or layout_table is not defined yet


### Checkpoint: interpret before you scroll on

Work through these with your group. The tables you just built contain a small ambush.

1. Which system wins on mean ROUGE-1 F? Which wins on ROUGE-2 and ROUGE-L? Why would a lead baseline do relatively better on the order sensitive metrics? Think about what the first sentence of every document has in common with the first sentence of every reference.
2. Now look at the standard deviations. Which system is more consistent, and by how much? If these summaries fed a customer facing feature at Cordwell, would you rather have a higher mean or a tighter spread?
3. The layout table splits ROUGE-L by document structure. Describe System A's behavior on `front_loaded` versus `buried` documents. Then describe System B's. Which corpus level number was hiding this?
4. The histogram shows the same story as the layout table in a different form. What shape do you see in System A's distribution, and what causes it?

The general lesson, which transfers to every eval you will ever run: **a corpus mean is a summary statistic, and summaries hide things.** Slicing an eval by a known document property is one of the highest leverage habits in model evaluation. Here the property was layout; in production it might be query type, document length, department, or customer segment.

## 5. Per example inspection

### Task 7: find and read the extremes

Averages framed the comparison; now read actual summaries. Metric numbers only become trustworthy after you have repeatedly checked them against your own judgment.

Implement two functions:

**`find_extreme_examples`** returns a dict with keys `A_best`, `A_worst`, `B_best`, `B_worst`, mapping to the `doc_id` of the document where each system scored its highest and lowest ROUGE-L F. If several documents tie for an extreme, take the smallest `doc_id`.

**`show_example`** takes the scores DataFrame and a `doc_id` and prints the document, the reference, both candidate summaries, and both ROUGE-L F scores, formatted for side by side human reading.

**Worked target output.** The dict shape (values here are placeholders) and the print format, shown for doc 0:

```text
>>> find_extreme_examples(corpus_with_scores)
{'A_best': <doc_id>, 'A_worst': <doc_id>, 'B_best': <doc_id>, 'B_worst': <doc_id>}

>>> show_example(corpus_with_scores, 0)
================ doc_id=0  dept=Flooring  layout=buried ================
DOCUMENT:
This policy update applies to the Flooring department at Cordwell Home and Hardware. ...

REFERENCE:
This policy update summarizes priorities for the Flooring team at Cordwell Home and Hardware. ...

SYSTEM A (ROUGE-L F = 0.3793):
This policy update applies to the Flooring department at Cordwell Home and Hardware. ...

SYSTEM B (ROUGE-L F = 0.2927):
A calcium chloride moisture test is required before any flooring goes over a concrete slab. ...
```

Your exact separator characters and spacing are up to you; the check only verifies `find_extreme_examples`.

In [14]:
# Task 7: extreme example inspection (STUDENT TODO)

def find_extreme_examples(scores_df):
    """Return the doc_ids where each system scores its ROUGE-L extremes.

    Contract:
      - Return a dict with exactly the keys A_best, A_worst, B_best,
        B_worst.
      - A_best is the doc_id of the row with the highest A_rl_f,
        A_worst the lowest; B_best and B_worst likewise for B_rl_f.
      - If several rows tie for an extreme, take the smallest doc_id.
      - Values must be plain ints.
    """
    raise NotImplementedError("Task 7")


def show_example(scores_df, doc_id):
    """Print one document with reference, candidates, and scores.

    Contract:
      - Locate the row whose doc_id column equals doc_id.
      - Print a header with doc_id, department, and layout, then the
        document text, the reference summary, and each system's
        summary labeled with its ROUGE-L F to four decimal places.
      - Formatting details are yours; optimize for side by side
        human reading.
    """
    raise NotImplementedError("Task 7")


try:
    inspect_ids = find_extreme_examples(corpus_with_scores)
    print(inspect_ids)
    for label, doc_id in inspect_ids.items():
        print()
        print(f"----- {label} -----")
        show_example(corpus_with_scores, doc_id)
except NotImplementedError as exc:
    print(f"Waiting on {exc}.")
except NameError:
    print("Finish Task 5 first: corpus_with_scores is not defined.")

check_task_7()

Finish Task 5 first: corpus_with_scores is not defined.
[TODO] Task 7 find_extreme_examples: inspect_ids is not defined yet


### Exercise: metric versus judgment

For each of the four extreme examples you printed:

1. Read the document, the reference, and both candidates as a human reviewer. Which candidate would you rather hand to a Cordwell associate?
2. Compare your preference to the ROUGE-L F numbers. Where do you and the metric agree? Where do you disagree, and what exactly is the metric rewarding or punishing in that case?

Patterns worth hunting for: a high scoring summary that leans on the opener sentence matching the reference opener nearly word for word, and a low scoring summary that actually carries the right facts but phrases them differently than the reference does.

## 6. Breaking ROUGE: hallucination and redundancy

ROUGE measures overlap with the reference. It never reads the source document, so it structurally cannot notice content that was invented, and it handles repetition in a very specific way. You are about to measure both blind spots precisely.

### Task 8: score the corrupted variants

The corruption materials are provided: three hallucination sentences that are obviously false about Cordwell, and eight fixed document ids so everyone in the room gets identical numbers.

Implement `build_variant_scores`. For each selected document, construct three candidates from that document's System B summary:

- `base`: the System B summary unchanged.
- `hallucinated`: the base summary, a space, then all three hallucination sentences joined by spaces.
- `redundant`: the base summary repeated three times, joined by spaces.

Score each variant against the reference and return a DataFrame with one row per document and variant, and columns `doc_id`, `variant`, `r1_precision`, `r1_recall`, `rl_recall`, `rl_f`. That is 24 rows.

**Worked target output.** The grouped means table your assembly cell will print, with the base row filled in:

```text
              r1_recall  r1_precision    rl_f
variant
base             0.5623        0.5087  0.3952
hallucinated        ...           ...     ...
redundant           ...           ...     ...
```

Before running it, make a prediction and write it down: what happens to recall when you append made up sentences? Most people predict it stays flat. The real answer is stranger.

In [15]:
# Task 8: hallucinated and redundant variants (STUDENT TODO)

# Corruption materials (provided). Fixed doc ids keep everyone's
# numbers identical for the group discussion.
HALLUCINATION_SENTENCES = [
    "Cordwell will soon offer free drone delivery on all orders over five dollars.",
    "Associates receive a complimentary espresso machine after every completed sale.",
    "The store basement contains a heated indoor pool for customer relaxation.",
]

SELECTED_DOC_IDS = [3, 17, 42, 88, 131, 204, 267, 350]


def build_variant_scores(scores_df):
    """Score base, hallucinated, and redundant variants per document.

    Contract:
      - For each doc_id in SELECTED_DOC_IDS, take that row's
        candidate_B as the base summary.
      - hallucinated = base + " " + the three HALLUCINATION_SENTENCES
        joined by single spaces.
      - redundant = the base summary repeated three times, joined by
        single spaces.
      - Score each of the three variants against the row's
        reference_summary with compute_rouge_scores.
      - Return a DataFrame with 24 rows and columns doc_id, variant,
        r1_precision, r1_recall, rl_recall, rl_f, where variant is
        one of base, hallucinated, redundant.
    """
    raise NotImplementedError("Task 8")


try:
    halluc_df = build_variant_scores(corpus_with_scores)
    display(halluc_df.groupby("variant")
            [["r1_recall", "r1_precision", "rl_f"]].mean().round(4))
except NotImplementedError as exc:
    print(f"Waiting on {exc}.")
except NameError:
    print("Finish Task 5 first: corpus_with_scores is not defined.")

check_task_8()

Finish Task 5 first: corpus_with_scores is not defined.
[TODO] Task 8 build_variant_scores: halluc_df is not defined yet


### Reflection: what did ROUGE catch, and which number caught it?

1. Recall went **up** for the hallucinated variant. Trace why: the made up sentences mention Cordwell, the store, and associates, and those generic words also appear in the reference. Adding false content added matching words. What does this tell you about using recall, or recall leaning F-measure, as your only summarization metric?
2. Precision dropped hard for hallucination and collapsed for redundancy. Explain each drop mechanically, in terms of what fraction of the candidate's words find a match in the reference.
3. ROUGE-L F dropped for both corrupted variants, but for the redundant variant the candidate is made entirely of correct sentences. Is the F drop detecting wrongness, or something else?
4. A colleague at Cordwell proposes gating a summarization deploy purely on mean ROUGE-L F against last quarter's references. Based on this section, write two sentences of feedback on that plan: one on what the gate would catch, one on what it would miss and which additional check you would add.

The production takeaway: ROUGE compares candidate to reference, never candidate to source. Faithfulness to the source needs its own check, whether that is precision monitoring, an entailment model, an LLM judge, or human review for high stakes content.

## 7. Stretch 1 (offline): the oracle ceiling

Every document carries an `oracle_summary`: by construction it contains exactly the content of the reference, expressed in the document's own words. Semantically it is a perfect summary. What does ROUGE give it?

Implement `compute_oracle_ceiling`: score every document's oracle summary against its reference, add the ROUGE-L F values to the DataFrame as a column named `oracle_rl_f`, and return the mean.

Then compare three numbers: the oracle mean, System A's mean, and System B's mean, all on ROUGE-L F. Also check the maximum oracle score across all 400 documents.

**What you will find.** The oracle mean lands around 0.69 and no document reaches 1.0. A summary that is perfect in content tops out about thirty points below the theoretical maximum, purely because of wording. Two consequences for practice: absolute ROUGE values are not interpretable against 1.0, and the honest way to report a system's ROUGE is relative to a measured ceiling such as a human or oracle baseline on the same test set. This is exactly why the slides warn against comparing ROUGE numbers across papers that used different references.

In [16]:
# Stretch 1 (optional, offline): the oracle ceiling (STUDENT TODO)

def compute_oracle_ceiling(scores_df):
    """Score every oracle summary against its reference.

    Contract:
      - For each row, compute ROUGE for oracle_summary against
        reference_summary and keep the rl_f value.
      - Add the values to scores_df as a new column oracle_rl_f.
      - Return the mean of that column as a float.
    """
    raise NotImplementedError("Stretch 1")


# Uncomment when ready:
# ceiling = compute_oracle_ceiling(corpus_with_scores)
# print(f"Oracle ceiling (mean ROUGE-L F):  {ceiling:.4f}")
# print(f"Oracle maximum across 400 docs:   "
#       f"{corpus_with_scores['oracle_rl_f'].max():.4f}")
# print(f"System A mean ROUGE-L F:          "
#       f"{corpus_with_scores['A_rl_f'].mean():.4f}")
# print(f"System B mean ROUGE-L F:          "
#       f"{corpus_with_scores['B_rl_f'].mean():.4f}")

## 8. Stretch 2 (requires network and extra installs): semantic similarity

ROUGE punished the oracle summaries for wording. A semantic metric should not. In this stretch you compare ROUGE-L against embedding based cosine similarity using sentence-transformers.

**Requirements beyond the core lab:** `torch` and `sentence-transformers` installed, plus one time network access to download the embedding model (about 90 MB). On the cohort Macs the model runs fine on MPS or CPU. If your environment cannot download models, skip this stretch; it is optional and nothing later depends on it.

The plan:

1. Load the `all-MiniLM-L6-v2` sentence transformer, placing it with the runtime device selector.
2. Embed every reference summary and every oracle summary, and compute their pairwise cosine similarity.
3. Compare: the oracle summaries that ROUGE-L scored around 0.53 to 0.83 should show embedding similarity well above that, tightly clustered near the top of the scale, because the meaning is identical even where the wording differs.
4. Scatter plot embedding similarity against `oracle_rl_f` from Stretch 1 to see how much of ROUGE's variation is wording noise that the semantic metric ignores.

The stub cell contains the full scaffold with the device selection convention. An instructor note in the solution covers expected results, since the exact similarity values depend on the downloaded model weights.

In [17]:
# Stretch 2 (optional, needs network and extra installs):
# semantic similarity versus ROUGE (STUDENT TODO)
#
# Requires: pip install torch sentence-transformers
# and one time network access to download all-MiniLM-L6-v2.
#
# The device selection scaffold below follows the cohort convention:
# the MPS fallback variable is set before torch is imported, and the
# device is chosen at runtime, never hard coded.

import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")


def pick_device():
    """Select the best available torch device at runtime."""
    try:
        import torch
        if torch.cuda.is_available():
            return "cuda"
        if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            return "mps"
    except Exception:
        pass
    return "cpu"


# TODO if you attempt this stretch:
#   1. Import SentenceTransformer from sentence_transformers inside a
#      try block, printing a clear skip message on ImportError.
#   2. Load "sentence-transformers/all-MiniLM-L6-v2" onto pick_device().
#   3. Encode corpus_with_scores["reference_summary"] and
#      corpus_with_scores["oracle_summary"] with normalize_embeddings=True.
#   4. Compute the row wise cosine similarity (a dot product per pair,
#      since the embeddings are normalized).
#   5. Compare the similarity distribution to oracle_rl_f from
#      Stretch 1: print both means, then scatter plot similarity
#      against oracle_rl_f.

## 9. Wrap up

Run the scoreboard cell below. Eight of eight is a completed lab; the stretches are extra.

What you built and what it taught:

- A sentence splitter, a lead baseline, and a TF-IDF selector: two systems whose ROUGE profiles disagree depending on which variant and which statistic you consult.
- A scoring function with the argument order handled correctly, which is a genuine production bug class, not a classroom nicety.
- A corpus level comparison that looked decisive until you sliced it by layout, at which point System A revealed a bimodal failure mode that the mean concealed.
- Direct measurements of ROUGE's blind spots: hallucination can raise recall, redundancy collapses precision, and neither is detected by comparing means alone.
- (Stretch) An oracle ceiling showing that perfect content tops out near 0.69 ROUGE-L F on this corpus, which recalibrates how you read every absolute ROUGE value from now on.

Carry three habits out of this lab: slice every eval by at least one document property, watch precision when hallucination is a risk, and never interpret an absolute ROUGE score without a ceiling estimate from the same references.

In [18]:
# 9.1 Final scoreboard

run_all_checks()

[TODO] Task 1 split_sentences
[TODO] Task 2 summarize_lead_k
[TODO] Task 3 summarize_tfidf
[TODO] Task 4 compute_rouge_scores
[TODO] Task 5 corpus_with_scores: corpus_with_scores is not defined yet
[TODO] Task 6 summary and layout tables: summary_table or layout_table is not defined yet
[TODO] Task 7 find_extreme_examples: inspect_ids is not defined yet
[TODO] Task 8 build_variant_scores: halluc_df is not defined yet

Scoreboard: 0 of 8 checks passing.
